In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pss5e12-main/__results__.html
/kaggle/input/pss5e12-main/submission_cv_0.7037958658238684.csv
/kaggle/input/pss5e12-main/xgb_feature_importance.csv
/kaggle/input/pss5e12-main/__notebook__.ipynb
/kaggle/input/pss5e12-main/__output__.json
/kaggle/input/pss5e12-main/custom.css
/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv
/kaggle/input/playground-series-s5e12/sample_submission.csv
/kaggle/input/playground-series-s5e12/train.csv
/kaggle/input/playground-series-s5e12/test.csv


In [2]:
class CONFIG:
    INPUT_DIR = '/kaggle/input/playground-series-s5e12'
    
    N_FOLDS = 5
    SEED = 42

    TARGET = 'diagnosed_diabetes'

config = CONFIG()

train = pd.read_csv(f'{config.INPUT_DIR}/train.csv')
test = pd.read_csv(f'{config.INPUT_DIR}/test.csv')
train_org = pd.read_csv('/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv')

train['source'] = 'train'
test['source'] = 'test'
train_org['source'] = 'original'

cols = train.columns
cols = [col for col in cols if col not in ['id']]

test[config.TARGET] = -1
train[cols] = train[cols].copy()
test[cols] = test[cols].copy()
train_org = train_org[cols].copy()
# test[config.TARGET] = -1


combine = pd.concat([train_org, train, test], axis=0, ignore_index=True)
submission = pd.read_csv(f'{config.INPUT_DIR}/sample_submission.csv')

In [3]:
# def eda(df, name):
#     print(f"Exploring {name} dataframe")
#     print('='*30)
#     print(f'\n NULL VALUES: {df.isnull().sum()}')
#     print('='*30)
#     print(f'\n {name} dataframe shape: {df.shape}')
#     print('='*30)
#     print(f'\n {name} dataframe numerical stats: {df.describe()}')
#     print('='*30)

# eda(train, 'TRAIN')
# eda(test, 'TEST')

In [4]:
FEATURES = [col for col in train.columns if col not in ['id', 'diagnosed_diabetes']]
CATS = train[FEATURES].select_dtypes(include='object').columns.to_list()
CATS = [col for col in CATS if col not in ['source']]
NUMS = train[FEATURES].select_dtypes(include=['int64', 'float64']).columns.to_list()

# combine = pd.concat([train, test])

In [5]:
CATS1 = []

for c in NUMS:
    n = f'{c}_cat'
    for df in [combine]:
        df[n] = df[c].astype('category')
        
    CATS1.append(n)

print(CATS1)

['age_cat', 'alcohol_consumption_per_week_cat', 'physical_activity_minutes_per_week_cat', 'diet_score_cat', 'sleep_hours_per_day_cat', 'screen_time_hours_per_day_cat', 'bmi_cat', 'waist_to_hip_ratio_cat', 'systolic_bp_cat', 'diastolic_bp_cat', 'heart_rate_cat', 'cholesterol_total_cat', 'hdl_cholesterol_cat', 'ldl_cholesterol_cat', 'triglycerides_cat', 'family_history_diabetes_cat', 'hypertension_history_cat', 'cardiovascular_history_cat']


In [6]:
CATS2 = []
SIZES = {}

for c in CATS+CATS1:
    n = f'{c}_enc'
    for df in [combine]:
        df[c] = df[c].astype('category')
        df[n], _ =  df[c].factorize()
        df[n] = df[n].astype('float32')
        s = df[n].max()+1

    CATS2.append(n)
    SIZES[n] = s

print(CATS2)
print('='*30)
print(f'CARDINALITY OF CATS2: {SIZES}')

['gender_enc', 'ethnicity_enc', 'education_level_enc', 'income_level_enc', 'smoking_status_enc', 'employment_status_enc', 'age_cat_enc', 'alcohol_consumption_per_week_cat_enc', 'physical_activity_minutes_per_week_cat_enc', 'diet_score_cat_enc', 'sleep_hours_per_day_cat_enc', 'screen_time_hours_per_day_cat_enc', 'bmi_cat_enc', 'waist_to_hip_ratio_cat_enc', 'systolic_bp_cat_enc', 'diastolic_bp_cat_enc', 'heart_rate_cat_enc', 'cholesterol_total_cat_enc', 'hdl_cholesterol_cat_enc', 'ldl_cholesterol_cat_enc', 'triglycerides_cat_enc', 'family_history_diabetes_cat_enc', 'hypertension_history_cat_enc', 'cardiovascular_history_cat_enc']
CARDINALITY OF CATS2: {'gender_enc': 3.0, 'ethnicity_enc': 5.0, 'education_level_enc': 4.0, 'income_level_enc': 5.0, 'smoking_status_enc': 3.0, 'employment_status_enc': 4.0, 'age_cat_enc': 73.0, 'alcohol_consumption_per_week_cat_enc': 11.0, 'physical_activity_minutes_per_week_cat_enc': 622.0, 'diet_score_cat_enc': 101.0, 'sleep_hours_per_day_cat_enc': 71.0, 'scr

In [7]:
# INTER = []

# for col1, col2 in combinations(CATS+CATS1, 2):
#     n = f'{col1}_{col2}_inter'

#     for df in [combine]:
#         df[n] = df[col1].astype(str) + '_' + df[col2].astype(str)
#         df[n] = df[n].astype('category')

#     INTER.append(n)

# # print(INTER)
# print('='*30)
# print('LENGTH OF INTER :', len(INTER))

In [8]:
# import warnings
# warnings.filterwarnings('ignore')

# NUMS_INTER = []

# # for col1, col2 in combinations(NUMS, 2):
# #     mul = f'{col1}_{col2}_mul'
# #     div = f'{col1}_{col2}_div'
# #     log1p = f'{col1}_{col2}_log1p'
# #     NUMS_INTER.extend([mul, div, log1p])
# #     for df in [combine]:
# #         df[mul] = df[col1] * df[col2]
# #         df[div] = df[col1] / (df[col2] + 1e-5)
# #         df[log1p] = np.log1p(df[f'{col1}_{col2}_mul'])

# for c in NUMS:
#     for t in ['_log']:
#         n = f'{c}_{t}'
#         NUMS_INTER.append(n)
#         for df in [combine]:
#             if t=='_log': df[n] = np.log1p(df[c])
#             elif t=='_poly2': df[n] = df[c]**2
#             else: continue
        

In [9]:
# print(len(NUMS_INTER))

In [10]:
pd.set_option('display.max_columns', None)
combine.head(5)

,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes,source,id,age_cat,alcohol_consumption_per_week_cat,physical_activity_minutes_per_week_cat,diet_score_cat,sleep_hours_per_day_cat,screen_time_hours_per_day_cat,bmi_cat,waist_to_hip_ratio_cat,systolic_bp_cat,diastolic_bp_cat,heart_rate_cat,cholesterol_total_cat,hdl_cholesterol_cat,ldl_cholesterol_cat,triglycerides_cat,family_history_diabetes_cat,hypertension_history_cat,cardiovascular_history_cat,gender_enc,ethnicity_enc,education_level_enc,income_level_enc,smoking_status_enc,employment_status_enc,age_cat_enc,alcohol_consumption_per_week_cat_enc,physical_activity_minutes_per_week_cat_enc,diet_score_cat_enc,sleep_hours_per_day_cat_enc,screen_time_hours_per_day_cat_enc,bmi_cat_enc,waist_to_hip_ratio_cat_enc,systolic_bp_cat_enc,diastolic_bp_cat_enc,heart_rate_cat_enc,cholesterol_total_cat_enc,hdl_cholesterol_cat_enc,ldl_cholesterol_cat_enc,triglycerides_cat_enc,family_history_diabetes_cat_enc,hypertension_history_cat_enc,cardiovascular_history_cat_enc
0,58,0,215,5.7,7.9,7.9,30.5,0.89,134,78,68,239,41,160,145,Male,Asian,Highschool,Lower-Middle,Never,Employed,0,0,0,1.0,original,NaN,58,0,215,5.7,7.9,7.9,30.5,0.89,134,78,68,239,41,160,145,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,48,1,143,6.7,6.5,8.7,23.1,0.80,129,76,67,116,55,50,30,Female,White,Highschool,Middle,Former,Employed,0,0,0,0.0,original,NaN,48,1,143,6.7,6.5,8.7,23.1,0.80,129,76,67,116,55,50,30,0,0,0,1.0,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0
2,60,1,57,6.4,10.0,8.1,22.2,0.81,115,73,74,213,66,99,36,Male,Hispanic,Highschool,Middle,Never,Unemployed,1,0,0,1.0,original,NaN,60,1,57,6.4,10.0,8.1,22.2,0.81,115,73,74,213,66,99,36,1,0,0,0.0,2.0,0.0,1.0,0.0,1.0,2.0,1.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,1.0,0.0,0.0
3,74,0,49,3.4,6.6,5.2,26.8,0.88,120,93,68,171,50,79,140,Female,Black,Highschool,Low,Never,Retired,0,0,0,1.0,original,NaN,74,0,49,3.4,6.6,5.2,26.8,0.88,120,93,68,171,50,79,140,0,0,0,1.0,3.0,0.0,2.0,0.0,2.0,3.0,0.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,0.0,3.0,3.0,3.0,3.0,0.0,0.0,0.0
4,46,1,109,7.2,7.4,5.0,21.2,0.78,92,67,67,210,52,125,160,Male,White,Graduate,Middle,Never,Retired,0,0,0,1.0,original,NaN,46,1,109,7.2,7.4,5.0,21.2,0.78,92,67,67,210,52,125,160,0,0,0,0.0,1.0,1.0,1.0,0.0,2.0,4.0,1.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,4.0,4.0,4.0,4.0,0.0,0.0,0.0


In [11]:
ROUND = []
rounding_levels = {'1s':0, '10s':-1}

for c in ['sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'diet_score', 'waist_to_hip_ratio']:
    for suffix, level in rounding_levels.items():
        new_col = f"{c}_ROUND_{suffix}"
        ROUND.append(new_col)
        for df in [combine]:
            df[new_col] = df[c].round(level).astype(int)

print(len(ROUND))

10


In [12]:
train_idx = combine['source'] == 'train'
test_idx = combine['source'] == 'test'
train_org_idx = combine['source'] == 'original'

# foundational_cols = [col for col in cols if col not in ['diagnosed_diabetes', 'source']]
# foundational_cols+=INTER
# cols+=INTER
# combine_truncated = combine[cols].copy()
# combine_truncated[foundational_cols] = combine_truncated[foundational_cols].astype(str).astype('category')

train_n = combine[train_idx].reset_index(drop=True).copy()
test_n = combine[test_idx].reset_index(drop=True).copy()
train_org_n = combine[train_org_idx].reset_index(drop=True).copy()

In [13]:
# print(len(foundational_cols))
# print('\n', foundational_cols) 

In [14]:
BINS = []
q_list = [10, 20, 30]
for col in NUMS:
    for q in q_list:
        col_name = f"{col}_bin{q}"

        _, bins = pd.qcut(
            train_org_n[col], q=q, retbins=True, labels=False,
            duplicates="drop"
        )

        train_org_n[col_name] = pd.cut(train_org_n[col], bins=bins, labels=False, include_lowest=True).astype(np.int8)
        train_n[col_name] = pd.cut(train_n[col], bins=bins, labels=False, include_lowest=True ).astype(np.int8)
        test_n[col_name] = pd.cut(test_n[col], bins=bins, labels=False, include_lowest=True).astype(np.int8)

        BINS.append(col_name)

print(f"{len(BINS)} k-bins discretization features created")

54 k-bins discretization features created


In [15]:
TE = []

for c in CATS+CATS1+BINS:
    tmp_mean = train_org_n.groupby(c)[config.TARGET].mean()
    tmp_count = train_org_n.groupby(c)[config.TARGET].size()

    n_m = f'TE_{c}_mean'
    n_c = f'TE_{c}_count'
    print(f'{n_m}, {n_c},', end=' ')
    tmp_mean.name = n_m
    tmp_count.name = n_c

    stats = (pd.concat([tmp_mean, tmp_count], axis=1).reset_index().rename(columns={'index': c}))

    train_org_n = train_org_n.merge(stats, on=c, how='left')
    train_n = train_n.merge(stats, on=c, how='left')
    test_n = test_n.merge(stats, on=c, how='left')

    TE.append(n_m)
    TE.append(n_c)

TE_gender_mean, TE_gender_count, TE_ethnicity_mean, TE_ethnicity_count, TE_education_level_mean, TE_education_level_count, TE_income_level_mean, TE_income_level_count, TE_smoking_status_mean, TE_smoking_status_count, TE_employment_status_mean, TE_employment_status_count, TE_age_cat_mean, TE_age_cat_count, TE_alcohol_consumption_per_week_cat_mean, TE_alcohol_consumption_per_week_cat_count, TE_physical_activity_minutes_per_week_cat_mean, TE_physical_activity_minutes_per_week_cat_count, TE_diet_score_cat_mean, TE_diet_score_cat_count, TE_sleep_hours_per_day_cat_mean, TE_sleep_hours_per_day_cat_count, TE_screen_time_hours_per_day_cat_mean, TE_screen_time_hours_per_day_cat_count, TE_bmi_cat_mean, TE_bmi_cat_count, TE_waist_to_hip_ratio_cat_mean, TE_waist_to_hip_ratio_cat_count, TE_systolic_bp_cat_mean, TE_systolic_bp_cat_count, TE_diastolic_bp_cat_mean, TE_diastolic_bp_cat_count, TE_heart_rate_cat_mean, TE_heart_rate_cat_count, TE_cholesterol_total_cat_mean, TE_cholesterol_total_cat_count, 

In [16]:
len(TE)

156

In [17]:
FEATURES_T = CATS + CATS1 + CATS2 + NUMS + ROUND + TE 
len(FEATURES_T)

232

In [18]:
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cols_to_encode, aggs=['mean'], cv=5, smooth='auto', drop_original=True):
        self.cols_to_encode = cols_to_encode
        self.aggs = aggs
        self.cv = cv
        self.smooth = smooth
        self.drop_original = drop_original
        self.mappings_ = {}
        self.global_stats_ = {}

    def fit(self, X, y):
        temp_df = X.copy()
        temp_df['target'] = y

        for agg_func in self.aggs:
            self.global_stats_[agg_func] = y.agg(agg_func)

        for col in self.cols_to_encode:
            self.mappings_[col] = {}
            for agg_func in self.aggs:
                mapping = temp_df.groupby(col)['target'].agg(agg_func)
                self.mappings_[col][agg_func] = mapping

        return self

    def transform(self, X):
        X_transformed = X.copy()
        for col in self.cols_to_encode:
            for agg_func in self.aggs:
                new_col_name = f'TE_{col}_{agg_func}'
                map_series = self.mappings_[col][agg_func]
    
                # 1. map, 2. cast to float, 3. fill NaN
                X_transformed[new_col_name] = (
                    X[col].map(map_series)
                         .astype(float)          # <-- key line
                         .fillna(self.global_stats_[agg_func])
                )
    
        if self.drop_original:                   # typo fixed
            X_transformed.drop(columns=self.cols_to_encode, inplace=True)
    
        return X_transformed

    def fit_transform(self, X, y):

        self.fit(X, y)
        encoded_features = pd.DataFrame(index=X.index)

        skf = StratifiedKFold(n_splits=self.cv, shuffle=True, random_state=42)

        for train_idx, val_idx in skf.split(X, y):
            X_train, y_train = X.iloc[train_idx], y.iloc[val_idx]
            X_val = X.iloc[val_idx]

            temp_df_train = X_train.copy()
            temp_df_train['target'] = y_train

            for col in self.cols_to_encode:
                for agg_func in self.aggs:
                    new_col_name = f'TE_{col}_{agg_func}'

                    fold_global_stats = y_train.agg(agg_func)
                    mapping = temp_df_train.groupby(col)['target'].agg(agg_func)

                    if agg_func == 'mean':
                        counts = temp_df_train.groupby(col)['target'].count()

                        m = self.smooth
                        if self.smooth == 'auto':
                            variance_between = mapping .var()
                            avg_variance_within = temp_df_train.groupby(col)['target'].mean()
                            if variance_between > 0:
                                m = avg_variance_within / variance_between

                            else:
                                m = 0

                        smoothed_mapping = (counts * mapping + m * fold_global_stats) / (counts + m)

                        encoded_values = X_val[col].map(smoothed_mapping)

                    else:
                        encoded_values = X_val[col].map(mapping)

                    encoded_features.loc[X_val.index, new_col_name] = encoded_values.fillna(fold_global_stats)

            X_transformed = X.copy()
            for col in encoded_features.columns:
                X_transformed[col] = encoded_features[col]

            if self.drop_original:
                X_transformed.drop(columns=self.cols_to_encode, inplace=True)

            return X_transformed
                    
                    

In [19]:
X = train_n[FEATURES_T].copy()
y = train_n[config.TARGET]

X_org = train_org_n[FEATURES_T].copy()
y_org = train_org_n[config.TARGET].copy()

test_n = test_n[FEATURES_T].copy()

# X = train_n[foundational_cols].copy()
# y = train_n[config.TARGET]

# X_org = train_org_n[foundational_cols].copy()
# y_org = train_org_n[config.TARGET]

# test_n = test_n[foundational_cols].copy()

In [20]:
# imp = pd.read_csv('/kaggle/input/pss5e12-main/xgb_feature_importance.csv')
# # imp_features = imp.iloc[:]['']
# imp = imp.iloc[:][imp['gain']>0.000507]
# main_features = imp.iloc[:]['feature_names']

In [21]:
# gain_arr = np.array(imp['gain'])
# gain_arr[:150]

In [22]:
from xgboost import XGBClassifier
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import roc_auc_score
import gc
import warnings
warnings.filterwarnings('ignore')

params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'max_depth': 5,
    'colsample_bytree': 0.5,
    'subsample': 0.8,
    'n_estimators': 10000,
    'learning_rate': 0.01,
    'early_stopping_rounds': 100,
    'random_state': 42,
    'n_jobs': -1,
    'device': 'cuda',
    'enable_categorical': True,
}

skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))
train = X.iloc[:-22000]
train_y = y.iloc[:-22000]
val = X.iloc[-22000:]
val_y = y.iloc[-22000:]
test_n = test_n.copy()
# X_org_n = X_org.copy()
# X.drop(columns=INTER, inplace=True)
# test_n.drop(columns=INTER, inplace=True)
fold_scores = []
for fold, (train_idx, val_idx) in enumerate(skf.split(val, val_y)):
    X_org_n = X_org.copy()
    y_org_n = y_org.copy()
    X_train, y_train = val.iloc[train_idx], val_y.iloc[train_idx]
    X_val, y_val = val.iloc[val_idx], val_y.iloc[val_idx]
    print(f'TRAINING SHAPE BEFORE DUPLICATION: {X_train.shape}')
    # for i in range(1):
    #     X_train = pd.concat([X_train, X_train], axis=0)
    #     y_train = pd.concat([y_train, y_train], axis=0)

    # for j in range(2):
    #     X_org_n = pd.concat([X_org_n, X_org_n], axis=0)
    #     y_org_n = pd.concat([y_org_n, y_org_n], axis=0)
    print(f'TRAINING SHAPE AFTER DUPLICATION: {X_train.shape}')
    X_train = pd.concat([train, X_org_n, X_train], axis=0, ignore_index=True)
    y_train = pd.concat([train_y, y_org_n, y_train], axis=0, ignore_index=True)
    print(f'TRAINING SHAPE :{X_train.shape}')
    print(f'VALIDATION SHAPE :{X_val.shape}')
    # X_val = pd.concat([X_val])
    # print(f'TRAINING SHAPE BEFORE ENCODING :{X_train.shape}')
    # TE = TargetEncoder(cols_to_encode=CATS, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=False)
    # X_train = TE.fit_transform(X_train, y_train)
    # X_val = TE.transform(X_val)
    # test_enc = TE.transform(test_n)

    # TE2 = TargetEncoder(cols_to_encode=ROUND, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=True)
    # X_train = TE2.fit_transform(X_train, y_train)
    # X_val = TE2.transform(X_val)
    # test_enc = TE2.transform(test_n)
    print(f'TRAINING SHAPE AFTER ENCODING :{X_train.shape}')
    model = XGBClassifier(**params)

    model.fit(X_train, y_train,
             eval_set=[(X_val, y_val)],
             verbose=1000)

    val_preds = model.predict_proba(X_val)[:,1]
    # oof_preds[val_idx] = val_preds

    fold_score = roc_auc_score(y_val, val_preds)
    fold_scores.append(fold_score)
    print(f'FOLD {fold} AUC: {fold_score:.4f}')
    test_preds +=  model.predict_proba(test_n)[:, 1] / config.N_FOLDS
    gc.collect()
    del X_train, y_train, X_org_n, y_org_n, X_val, y_val

# overall_auc = roc_auc_score(y, oof_preds)
overall_auc = np.mean(fold_scores)
print('='*30)
print(f"Overall OOF AUC: {overall_auc:.4f}")
print('='*30)

TRAINING SHAPE BEFORE DUPLICATION: (17600, 232)
TRAINING SHAPE AFTER DUPLICATION: (17600, 232)
TRAINING SHAPE :(795600, 232)
VALIDATION SHAPE :(4400, 232)
TRAINING SHAPE AFTER ENCODING :(795600, 232)
[0]	validation_0-auc:0.67173
[1000]	validation_0-auc:0.69283
[1277]	validation_0-auc:0.69301
FOLD 0 AUC: 0.6931
TRAINING SHAPE BEFORE DUPLICATION: (17600, 232)
TRAINING SHAPE AFTER DUPLICATION: (17600, 232)
TRAINING SHAPE :(795600, 232)
VALIDATION SHAPE :(4400, 232)
TRAINING SHAPE AFTER ENCODING :(795600, 232)
[0]	validation_0-auc:0.68732
[1000]	validation_0-auc:0.71183
[1089]	validation_0-auc:0.71175
FOLD 1 AUC: 0.7119
TRAINING SHAPE BEFORE DUPLICATION: (17600, 232)
TRAINING SHAPE AFTER DUPLICATION: (17600, 232)
TRAINING SHAPE :(795600, 232)
VALIDATION SHAPE :(4400, 232)
TRAINING SHAPE AFTER ENCODING :(795600, 232)
[0]	validation_0-auc:0.67759
[928]	validation_0-auc:0.70201
FOLD 2 AUC: 0.7022
TRAINING SHAPE BEFORE DUPLICATION: (17600, 232)
TRAINING SHAPE AFTER DUPLICATION: (17600, 232)
TR

# lgb

In [23]:
# from lightgbm import LGBMClassifier
# from sklearn.model_selection import KFold, StratifiedKFold
# from sklearn.metrics import roc_auc_score
# import warnings
# warnings.filterwarnings('ignore')

# params = {
#     'objective': 'binary',
#     'metric': 'auc',
#     'max_depth': 5,
#     'colsample_bytree': 0.5,
#     'subsample': 0.8,
#     'n_estimators': 10000,
#     'learning_rate': 0.01,
#     'random_state': 42,
#     'n_jobs': -1,
#     'device_type': 'gpu',  # For GPU training. Use 'cpu' if GPU not available
#     'enable_categorical': True,
#     'early_stopping_rounds': 200,
#     'verbosity': -1,  # Suppress default LightGBM output
# }

# skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.SEED)

# oof_preds = np.zeros(len(X))
# test_preds = np.zeros(len(test))
# train = X.iloc[:-22000]
# train_y = y.iloc[:-22000]
# val = X.iloc[-22000:]
# val_y = y.iloc[-22000:]
# test_n = test_n.copy()
# X_org = X_org.copy()
# # y_org should be defined somewhere in your notebook

# fold_scores = []
# for fold, (val_idx, train_idx) in enumerate(skf.split(val, val_y)):
#     X_train, y_train = val.iloc[train_idx], val_y.iloc[train_idx]
#     X_val, y_val = val.iloc[val_idx], val_y.iloc[val_idx]
#     print(f'TRAINING SHAPE BEFORE DUPLICATION: {X_train.shape}')
#     for i in range(2):
#         X_train = pd.concat([X_train, X_train], axis=0)
#         y_train = pd.concat([y_train, y_train], axis=0)
#     print(f'TRAINING SHAPE AFTER DUPLICATION: {X_train.shape}')
#     X_train = pd.concat([train, X_org, X_train], axis=0, ignore_index=True)
#     y_train = pd.concat([train_y, y_org, y_train], axis=0, ignore_index=True)
    
#     print(f'TRAINING SHAPE: {X_train.shape}')
#     print(f'VALIDATION SHAPE: {X_val.shape}')
    
#     # Target encoding sections (commented out)
#     # TE = TargetEncoder(cols_to_encode=CATS, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=False)
#     # X_train = TE.fit_transform(X_train, y_train)
#     # X_val = TE.transform(X_val)
#     # test_enc = TE.transform(test_n)
#     # 
#     # TE2 = TargetEncoder(cols_to_encode=ROUND, aggs=['mean', 'count'], cv=5, smooth='auto', drop_original=True)
#     # X_train = TE2.fit_transform(X_train, y_train)
#     # X_val = TE2.transform(X_val)
#     # test_enc = TE2.transform(test_n)
    
#     print(f'TRAINING SHAPE AFTER ENCODING: {X_train.shape}')
    
#     model = LGBMClassifier(**params)
    
#     model.fit(X_train, y_train,
#               eval_set=[(X_val, y_val)],
#               # early_stopping_rounds=100,
#               # verbose=1000
#              )  # Print every 100 iterations

#     val_preds = model.predict_proba(X_val)[:, 1]
#     # oof_preds[val_idx] = val_preds

#     fold_score = roc_auc_score(y_val, val_preds)
#     fold_scores.append(fold_score)
#     print(f'FOLD {fold} AUC: {fold_score:.4f}')
    
#     test_preds += model.predict_proba(test_n)[:, 1] / config.N_FOLDS

# overall_auc = np.mean(fold_scores)
# print('='*30)
# print(f"Overall OOF AUC: {overall_auc:.4f}")
# print('='*30)

In [24]:
submission[config.TARGET] = test_preds
submission.to_csv(f'submission_cv_{overall_auc}.csv', index=False)

In [25]:
imp_n = pd.DataFrame(
    {
        'feature_names': model.feature_names_in_,
        'gain': model.feature_importances_
    }
).sort_values('gain', ascending=False).reset_index(drop=True)

imp_n.to_csv('xgb_feature_importance.csv', index=False)
imp_n.head(10)

,feature_names,gain
0,TE_family_history_diabetes_cat_mean,0.137865
1,family_history_diabetes_cat,0.099944
2,family_history_diabetes_cat_enc,0.095273
3,family_history_diabetes,0.080605
4,TE_family_history_diabetes_cat_count,0.076158
5,TE_physical_activity_minutes_per_week_cat_mean,0.044415
6,physical_activity_minutes_per_week,0.033951
7,TE_physical_activity_minutes_per_week_bin30_mean,0.032399
8,TE_physical_activity_minutes_per_week_bin20_mean,0.022437
9,TE_physical_activity_minutes_per_week_bin10_mean,0.018301


In [26]:
# most_imp = imp.iloc[:65]['feature_names']
# list(most_imp)
# most_imp

In [27]:
# original_feats = CATS+NUMS
# imp['original'] = imp['feature_names'].isin(original_feats)
# imp_2 = imp[imp['original']]
# imp_2.head(10)

In [28]:
# print("CATS :", CATS)
# print('='*20)
# print("NUMS :", NUMS)